# 🔥 Spark ↔ SeaweedFS — Test por fases
Ejecuta cada celda una a una y verifica que funciona antes de pasar a la siguiente.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("test-seaweedfs") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://seaweedfs:8333") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

print("✅ SparkSession creada")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 12:19:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ SparkSession creada


## FASE 1 — Configuración

In [2]:
S3_ENDPOINT       = 'http://localhost:8333'
AWS_ACCESS_KEY_ID = 'admin'
AWS_SECRET_ACCESS = 'admin'

PERSISTENT_BUCKET = 'persistent-landing-zone'
TRUSTED_BUCKET    = 'trusted-zone'
PREFIX            = 'huggingface/kanishka089/'

persistent_path = f's3a://{PERSISTENT_BUCKET}/{PREFIX}'
trusted_path    = f's3a://{TRUSTED_BUCKET}/{PREFIX}'

print('Config OK')
print(f'  persistent_path : {persistent_path}')
print(f'  trusted_path    : {trusted_path}')

Config OK
  persistent_path : s3a://persistent-landing-zone/huggingface/kanishka089/
  trusted_path    : s3a://trusted-zone/huggingface/kanishka089/


In [3]:
BUCKET = "persistent-landing-zone"
PREFIX = "huggingface/kanishka089/"

ruta = f"s3a://{BUCKET}/{PREFIX}"
print(f"📖 Leyendo de: {ruta}")

df = spark.read.parquet(ruta)
print("✅ Lectura correcta!")
df.printSchema()
print(f"Total filas: {df.count()}")

📖 Leyendo de: s3a://persistent-landing-zone/huggingface/kanishka089/


26/05/03 12:19:49 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

✅ Lectura correcta!
root
 |-- Humidity: long (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Heat Index: double (nullable = true)
 |-- Atmospheric pressure: double (nullable = true)
 |-- Altitude: double (nullable = true)
 |-- Rain: double (nullable = true)



[Stage 1:>                                                          (0 + 1) / 1]

Total filas: 1379


In [2]:
import sys
import os

sys.path.insert(0, '/opt/spark/work-dir/trusted-zone')

os.environ['S3A_ENDPOINT'] = 'http://seaweedfs:8333'
os.environ['S3A_ACCESS_KEY'] = 'admin'
os.environ['S3A_SECRET_KEY'] = 'admin'
os.environ['SPARK_MASTER'] = 'spark://spark-master:7077'
os.environ['PERSISTENT_BUCKET'] = 'persistent-landing-zone'
os.environ['TRUSTED_BUCKET'] = 'trusted-zone'

In [3]:
from trusted_zone import landing_base_uri, trusted_base_uri
print("Landing:", landing_base_uri())
print("Trusted:", trusted_base_uri())

Landing: s3a://persistent-landing-zone/
Trusted: s3a://trusted-zone/


In [5]:
from pyspark.sql import SparkSession

# Tanca qualsevol sessió activa
existing = SparkSession.getActiveSession()
if existing:
    existing.stop()
    print("Sessió anterior tancada")

In [6]:
from trusted_zone import main
main()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 12:22:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/03 12:22:06 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


IllegalArgumentException: path must be absolute